## Self-Supervised Speech Recognition - Toronto Dataset

In [1]:
%load_ext autoreload
%autoreload 2


In [2]:
from pathlib import Path
import pandas as pd
import torch
from src.dataset import TimitDataset, build_timit_df, map_phonemes, DataCollatorCTCWithPadding
import os
import json
import lightning as L
from lightning.pytorch.loggers import CSVLogger
from lightning.pytorch.callbacks import ModelCheckpoint
from transformers import (
  Wav2Vec2PhonemeCTCTokenizer,
  Wav2Vec2FeatureExtractor,
  Wav2Vec2Processor,
  Data2VecAudioForCTC,
)
from torch.utils.data import DataLoader
import evaluate
from sklearn.model_selection import StratifiedGroupKFold

os.environ["PYTORCH_ENABLE_MPS_FALLBACK"] = "1"
DATA_DIR = Path("data")
TIMIT = DATA_DIR / Path("timit")
TIMIT_SR = 16000

PHONE_MAP = {
    "aa": "aa", "ae": "ae", "ah": "ah", "ao": "aa", "aw": "aw",
    "ax": "ah", "ax-h": "ah", "axr": "er", "ay": "ay",
    "b": "b", "bcl": "sil",
    "ch": "ch",
    "d": "d", "dcl": "sil", "dh": "dh", "dx": "dx",
    "eh": "eh", "el": "l", "em": "m", "en": "n", "eng": "ng",
    "epi": "sil", "er": "er", "ey": "ey",
    "f": "f",
    "g": "g", "gcl": "sil",
    "h#": "sil", "hh": "hh", "hv": "hh",
    "ih": "ih", "ix": "ih", "iy": "iy",
    "jh": "jh",
    "k": "k", "kcl": "sil",
    "l": "l",
    "m": "m",
    "n": "n", "ng": "ng", "nx": "n",
    "ow": "ow", "oy": "oy",
    "p": "p", "pau": "sil", "pcl": "sil",
    "q": None,
    "r": "r",
    "s": "s", "sh": "sh",
    "t": "t", "tcl": "sil", "th": "th",
    "uh": "uh", "uw": "uw", "ux": "uw",
    "v": "v",
    "w": "w",
    "y": "y",
    "z": "z", "zh": "sh",
}

assert len(PHONE_MAP) == 61

TIMIT_META_TEST = TIMIT / "test_data.csv"
TIMIT_META_TRAIN = TIMIT / "train_data.csv"

TIMIT_TEST = TIMIT / "data/test"
TIMIT_TRAIN = TIMIT / "data/train"

device = "cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu"


In [3]:
PARQUET_TEST = TIMIT / "test.parquet"
PARQUET_TRAIN = TIMIT / "train.parquet"


if not PARQUET_TEST.exists():
  timit_test_meta  = pd.read_csv(TIMIT_META_TEST)
  timit_test_df  = build_timit_df(timit_test_meta, TIMIT / "data", TIMIT_SR)
  timit_test_df["phonemes_39"] = timit_test_df["phonemes"].apply(lambda x: map_phonemes(x, PHONE_MAP))
  timit_test_df = timit_test_df[timit_test_df["sentence_type"] != "SA"].copy()
  timit_test_df.to_parquet(PARQUET_TEST)
else:
  timit_test_df = pd.read_parquet(PARQUET_TEST)

if not PARQUET_TRAIN.exists():
  timit_train_meta = pd.read_csv(TIMIT_META_TRAIN)
  timit_train_df = build_timit_df(timit_train_meta, TIMIT / "data", TIMIT_SR)
  timit_train_df["phonemes_39"] = timit_train_df["phonemes"].apply(lambda x: map_phonemes(x, PHONE_MAP))
  timit_train_df = timit_train_df[timit_train_df["sentence_type"] != "SA"].copy()
  timit_train_df["strat"] = timit_train_df["dialect_region"] + "_" + timit_train_df["gender"]
  timit_train_df.to_parquet(PARQUET_TRAIN)
else:
  timit_train_df = pd.read_parquet(PARQUET_TRAIN)


In [4]:
VOCAB = TIMIT / "vocab.json"

vocab_dict = {v: i for i, v in enumerate(sorted(set(PHONE_MAP.values()) - {None}))}
vocab_dict["<unk>"] = len(vocab_dict)
vocab_dict["<pad>"] = len(vocab_dict)

with open(VOCAB, "w", encoding="utf-8") as f:
  json.dump(vocab_dict, f, ensure_ascii=False)


In [5]:
class Data2VecCTC(L.LightningModule):
  def __init__(self, model, processor):
    super().__init__()
    self.model = model
    self.processor = processor
    self.per = evaluate.load("wer")

  def on_train_epoch_start(self):
    self.model.train()

  def training_step(self, batch, _):
    loss = self.model(**batch).loss
    self.log("train_loss", loss, on_step=False, on_epoch=True, prog_bar=True)
    return loss

  def validation_step(self, batch, _):
    out = self.model(**batch)
    self.log("val_loss", out.loss, prog_bar=True)

    pred_ids = torch.argmax(out.logits, dim=-1)

    labels = batch["labels"].clone()
    labels[labels == -100] = self.processor.tokenizer.pad_token_id

    preds = self.processor.batch_decode(pred_ids)
    refs = self.processor.batch_decode(labels, group_tokens=False)

    self.per.add_batch(predictions=preds, references=refs)

  def on_validation_epoch_end(self):
    self.log("val_per", self.per.compute(), prog_bar=True)

  def configure_optimizers(self):
    head_params = self.model.lm_head.parameters()
    encoder_params = [p for p in self.model.data2vec_audio.parameters() if p.requires_grad]
    return torch.optim.AdamW([
      {"params": head_params, "lr": 1e-4},
      {"params": encoder_params, "lr": 1e-5},
    ])



In [6]:
BACKBONE_MODEL = "facebook/data2vec-audio-base"

tokenizer = Wav2Vec2PhonemeCTCTokenizer(VOCAB, do_phonemize=False)

feature_extractor = Wav2Vec2FeatureExtractor.from_pretrained(BACKBONE_MODEL)
processor = Wav2Vec2Processor(feature_extractor=feature_extractor, tokenizer=tokenizer)


In [7]:
collator = DataCollatorCTCWithPadding(processor=processor)

sgkf = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=42)
train_ids, val_ids = next(sgkf.split(timit_train_df, timit_train_df["strat"], groups=timit_train_df["speaker_id"]))
timit_tr_df, timit_val_df = timit_train_df.iloc[train_ids], timit_train_df.iloc[val_ids]

timit_train_ds = TimitDataset(timit_tr_df, processor, features_column="input_values")
timit_val_ds = TimitDataset(timit_val_df, processor, features_column="input_values")
timit_test_ds = TimitDataset(timit_test_df, processor, features_column="input_values")

timit_train_dl = DataLoader(
  timit_train_ds,
  batch_size=8,
  shuffle=True,
  collate_fn=collator,
  num_workers=11,
  pin_memory=True,
  persistent_workers=True,
)

timit_val_dl = DataLoader(
  timit_val_ds,
  batch_size=8,
  shuffle=False,
  collate_fn=collator,
  num_workers=11,
  pin_memory=True,
)

timit_test_dl = DataLoader(
  timit_test_ds,
  batch_size=8,
  shuffle=False,
  collate_fn=collator,
  num_workers=11,
  pin_memory=True,
)

model = Data2VecAudioForCTC.from_pretrained(
  BACKBONE_MODEL,
  vocab_size=len(vocab_dict),
  pad_token_id=tokenizer.pad_token_id,
  ctc_zero_infinity=True,
  ctc_loss_reduction="mean",
  attn_implementation="sdpa",
)

model.freeze_feature_encoder()

logger = CSVLogger("logs", name="timit_data2vec_finetune_val_0_layers")
m = Data2VecCTC(model, processor)

checkpoint = ModelCheckpoint(
  monitor="val_per",
  mode="min",
  save_top_k=1,
  filename="{epoch}-{val_per:.3f}",
)

trainer = L.Trainer(
  accelerator=device,
  logger=logger,
  max_epochs=5,
  precision="bf16-mixed",
  callbacks=[checkpoint],
  gradient_clip_val=1.0,
)

model.train()

for param in model.data2vec_audio.parameters():
    param.requires_grad = False

trainer.fit(m, train_dataloaders=timit_train_dl, val_dataloaders=timit_val_dl)
m.eval()
trainer.validate(m, dataloaders=timit_test_dl, ckpt_path="best")


Loading weights:   0%|          | 0/230 [00:00<?, ?it/s]

[transformers] Data2VecAudioForCTC LOAD REPORT from: facebook/data2vec-audio-base
Key            | Status  | 
---------------+---------+-
lm_head.bias   | MISSING | 
lm_head.weight | MISSING | 

Notes:
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
Using bfloat16 Automatic Mixed Precision (AMP)
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
You are using a CUDA device ('NVIDIA GeForce RTX 4070 SUPER') that has Tensor Cores. To properly utilize them, you should set `torch.set_float32_matmul_precision('medium' | 'high')` which will trade-off precision for performance. For more details, read https://pytorch.org/docs/stable/generated/torch.set_float32_m

┏━━━┳━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃   ┃ Name  ┃ Type                ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━╇━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0 │ model │ Data2VecAudioForCTC │ 93.2 M │ train │     0 │
└───┴───────┴─────────────────────┴────────┴───────┴───────┘

Trainable params: 31.5 K                                                                                           
Non-trainable params: 93.2 M                                                                                       
Total params: 93.2 M                                                                                               
Total estimated model params size (MB): 372.783                                                                    
Modules in train mode: 249                                                                                         
Modules in eval mode: 0                                                                                            
Total FLOPs: 0

Output()

...

`Trainer.fit` stopped: `max_epochs=5` reached.


Restoring states from the checkpoint path at logs/timit_data2vec_finetune_val_0_layers/version_3/checkpoints/epoch=4-val_per=0.507.ckpt
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
Loaded model weights from the checkpoint at logs/timit_data2vec_finetune_val_0_layers/version_3/checkpoints/epoch=4-val_per=0.507.ckpt


Output()

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃      Validate metric      ┃       DataLoader 0        ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│         val_loss          │     3.26299786567688      │
│          val_per          │     0.51765376329422      │
└───────────────────────────┴───────────────────────────┘

[{'val_loss': 3.26299786567688, 'val_per': 0.51765376329422}]

In [8]:
collator = DataCollatorCTCWithPadding(processor=processor)

sgkf = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=42)
train_ids, val_ids = next(sgkf.split(timit_train_df, timit_train_df["strat"], groups=timit_train_df["speaker_id"]))
timit_tr_df, timit_val_df = timit_train_df.iloc[train_ids], timit_train_df.iloc[val_ids]

timit_train_ds = TimitDataset(timit_tr_df, processor, features_column="input_values")
timit_val_ds = TimitDataset(timit_val_df, processor, features_column="input_values")
timit_test_ds = TimitDataset(timit_test_df, processor, features_column="input_values")

timit_train_dl = DataLoader(
  timit_train_ds,
  batch_size=8,
  shuffle=True,
  collate_fn=collator,
  num_workers=11,
  pin_memory=True,
  persistent_workers=True,
)

timit_val_dl = DataLoader(
  timit_val_ds,
  batch_size=8,
  shuffle=False,
  collate_fn=collator,
  num_workers=11,
  pin_memory=True,
)

timit_test_dl = DataLoader(
  timit_test_ds,
  batch_size=8,
  shuffle=False,
  collate_fn=collator,
  num_workers=11,
  pin_memory=True,
)

model = Data2VecAudioForCTC.from_pretrained(
  BACKBONE_MODEL,
  vocab_size=len(vocab_dict),
  pad_token_id=tokenizer.pad_token_id,
  ctc_zero_infinity=True,
  ctc_loss_reduction="mean",
  attn_implementation="sdpa",
)

model.freeze_feature_encoder()

logger = CSVLogger("logs", name="timit_data2vec_finetune_val_4_layers")
m = Data2VecCTC(model, processor)

checkpoint = ModelCheckpoint(
  monitor="val_per",
  mode="min",
  save_top_k=1,
  filename="{epoch}-{val_per:.3f}",
)

trainer = L.Trainer(
  accelerator=device,
  logger=logger,
  max_epochs=5,
  callbacks=[checkpoint],
  gradient_clip_val=1.0,
)

model.train()

for param in model.data2vec_audio.parameters():
    param.requires_grad = False

n_unfrozen = 4
for layer in model.data2vec_audio.encoder.layers[-n_unfrozen:]:
  for param in layer.parameters():
    param.requires_grad = True

trainer.fit(m, train_dataloaders=timit_train_dl, val_dataloaders=timit_val_dl)
m.eval()
trainer.validate(m, dataloaders=timit_test_dl, ckpt_path="best")


Loading weights:   0%|          | 0/230 [00:00<?, ?it/s]

[transformers] Data2VecAudioForCTC LOAD REPORT from: facebook/data2vec-audio-base
Key            | Status  | 
---------------+---------+-
lm_head.bias   | MISSING | 
lm_head.weight | MISSING | 

Notes:
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


┏━━━┳━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃   ┃ Name  ┃ Type                ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━╇━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0 │ model │ Data2VecAudioForCTC │ 93.2 M │ train │     0 │
└───┴───────┴─────────────────────┴────────┴───────┴───────┘

Trainable params: 28.4 M                                                                                           
Non-trainable params: 64.8 M                                                                                       
Total params: 93.2 M                                                                                               
Total estimated model params size (MB): 372.783                                                                    
Modules in train mode: 249                                                                                         
Modules in eval mode: 0                                                                                            
Total FLOPs: 0

Output()

...

`Trainer.fit` stopped: `max_epochs=5` reached.


Restoring states from the checkpoint path at logs/timit_data2vec_finetune_val_4_layers/version_1/checkpoints/epoch=4-val_per=0.179.ckpt
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
Loaded model weights from the checkpoint at logs/timit_data2vec_finetune_val_4_layers/version_1/checkpoints/epoch=4-val_per=0.179.ckpt


Output()

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃      Validate metric      ┃       DataLoader 0        ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│         val_loss          │    0.43438175320625305    │
│          val_per          │    0.16976001858711243    │
└───────────────────────────┴───────────────────────────┘

[{'val_loss': 0.43438175320625305, 'val_per': 0.16976001858711243}]

In [9]:
collator = DataCollatorCTCWithPadding(processor=processor)

sgkf = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=42)
train_ids, val_ids = next(sgkf.split(timit_train_df, timit_train_df["strat"], groups=timit_train_df["speaker_id"]))
timit_tr_df, timit_val_df = timit_train_df.iloc[train_ids], timit_train_df.iloc[val_ids]

timit_train_ds = TimitDataset(timit_tr_df, processor, features_column="input_values")
timit_val_ds = TimitDataset(timit_val_df, processor, features_column="input_values")
timit_test_ds = TimitDataset(timit_test_df, processor, features_column="input_values")

timit_train_dl = DataLoader(
  timit_train_ds,
  batch_size=8,
  shuffle=True,
  collate_fn=collator,
  num_workers=11,
  pin_memory=True,
  persistent_workers=True,
)

timit_val_dl = DataLoader(
  timit_val_ds,
  batch_size=8,
  shuffle=False,
  collate_fn=collator,
  num_workers=11,
  pin_memory=True,
)

timit_test_dl = DataLoader(
  timit_test_ds,
  batch_size=8,
  shuffle=False,
  collate_fn=collator,
  num_workers=11,
  pin_memory=True,
)

model = Data2VecAudioForCTC.from_pretrained(
  BACKBONE_MODEL,
  vocab_size=len(vocab_dict),
  pad_token_id=tokenizer.pad_token_id,
  ctc_zero_infinity=True,
  ctc_loss_reduction="mean",
  attn_implementation="sdpa",
)

model.freeze_feature_encoder()

logger = CSVLogger("logs", name="timit_data2vec_finetune_val_6_layers")
m = Data2VecCTC(model, processor)

checkpoint = ModelCheckpoint(
  monitor="val_per",
  mode="min",
  save_top_k=1,
  filename="{epoch}-{val_per:.3f}",
)

trainer = L.Trainer(
  accelerator=device,
  logger=logger,
  max_epochs=5,
  callbacks=[checkpoint],
  gradient_clip_val=1.0,
)

model.train()

for param in model.data2vec_audio.parameters():
    param.requires_grad = False

n_unfrozen = len(model.data2vec_audio.encoder.layers) // 2
for layer in model.data2vec_audio.encoder.layers[-n_unfrozen:]:
  for param in layer.parameters():
    param.requires_grad = True

trainer.fit(m, train_dataloaders=timit_train_dl, val_dataloaders=timit_val_dl)
m.eval()
trainer.validate(m, dataloaders=timit_test_dl, ckpt_path="best")


Loading weights:   0%|          | 0/230 [00:00<?, ?it/s]

[transformers] Data2VecAudioForCTC LOAD REPORT from: facebook/data2vec-audio-base
Key            | Status  | 
---------------+---------+-
lm_head.bias   | MISSING | 
lm_head.weight | MISSING | 

Notes:
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


┏━━━┳━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃   ┃ Name  ┃ Type                ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━╇━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0 │ model │ Data2VecAudioForCTC │ 93.2 M │ train │     0 │
└───┴───────┴─────────────────────┴────────┴───────┴───────┘

Trainable params: 42.6 M                                                                                           
Non-trainable params: 50.6 M                                                                                       
Total params: 93.2 M                                                                                               
Total estimated model params size (MB): 372.783                                                                    
Modules in train mode: 249                                                                                         
Modules in eval mode: 0                                                                                            
Total FLOPs: 0

Output()

...

`Trainer.fit` stopped: `max_epochs=5` reached.


Restoring states from the checkpoint path at logs/timit_data2vec_finetune_val_6_layers/version_3/checkpoints/epoch=3-val_per=0.158.ckpt
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
Loaded model weights from the checkpoint at logs/timit_data2vec_finetune_val_6_layers/version_3/checkpoints/epoch=3-val_per=0.158.ckpt


Output()

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃      Validate metric      ┃       DataLoader 0        ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│         val_loss          │    0.4370085299015045     │
│          val_per          │    0.1563226580619812     │
└───────────────────────────┴───────────────────────────┘

[{'val_loss': 0.4370085299015045, 'val_per': 0.1563226580619812}]

In [10]:
collator = DataCollatorCTCWithPadding(processor=processor)

sgkf = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=42)
train_ids, val_ids = next(sgkf.split(timit_train_df, timit_train_df["strat"], groups=timit_train_df["speaker_id"]))
timit_tr_df, timit_val_df = timit_train_df.iloc[train_ids], timit_train_df.iloc[val_ids]

timit_train_ds = TimitDataset(timit_tr_df, processor, features_column="input_values")
timit_val_ds = TimitDataset(timit_val_df, processor, features_column="input_values")
timit_test_ds = TimitDataset(timit_test_df, processor, features_column="input_values")

timit_train_dl = DataLoader(
  timit_train_ds,
  batch_size=8,
  shuffle=True,
  collate_fn=collator,
  num_workers=11,
  pin_memory=True,
  persistent_workers=True,
)

timit_val_dl = DataLoader(
  timit_val_ds,
  batch_size=8,
  shuffle=False,
  collate_fn=collator,
  num_workers=11,
  pin_memory=True,
)

timit_test_dl = DataLoader(
  timit_test_ds,
  batch_size=8,
  shuffle=False,
  collate_fn=collator,
  num_workers=11,
  pin_memory=True,
)

model = Data2VecAudioForCTC.from_pretrained(
  BACKBONE_MODEL,
  vocab_size=len(vocab_dict),
  pad_token_id=tokenizer.pad_token_id,
  ctc_zero_infinity=True,
  ctc_loss_reduction="mean",
  attn_implementation="sdpa",
)

model.freeze_feature_encoder()

logger = CSVLogger("logs", name="timit_data2vec_finetune_val_12_layers")
m = Data2VecCTC(model, processor)

checkpoint = ModelCheckpoint(
  monitor="val_per",
  mode="min",
  save_top_k=1,
  filename="{epoch}-{val_per:.3f}",
)

trainer = L.Trainer(
  accelerator=device,
  logger=logger,
  max_epochs=5,
  callbacks=[checkpoint],
  gradient_clip_val=1.0,
)

model.train()

for param in model.data2vec_audio.parameters():
    param.requires_grad = False

for layer in model.data2vec_audio.encoder.layers:
  for param in layer.parameters():
    param.requires_grad = True

trainer.fit(m, train_dataloaders=timit_train_dl, val_dataloaders=timit_val_dl)
m.eval()
trainer.validate(m, dataloaders=timit_test_dl, ckpt_path="best")


Loading weights:   0%|          | 0/230 [00:00<?, ?it/s]

[transformers] Data2VecAudioForCTC LOAD REPORT from: facebook/data2vec-audio-base
Key            | Status  | 
---------------+---------+-
lm_head.bias   | MISSING | 
lm_head.weight | MISSING | 

Notes:
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


┏━━━┳━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃   ┃ Name  ┃ Type                ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━╇━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0 │ model │ Data2VecAudioForCTC │ 93.2 M │ train │     0 │
└───┴───────┴─────────────────────┴────────┴───────┴───────┘

Trainable params: 85.1 M                                                                                           
Non-trainable params: 8.1 M                                                                                        
Total params: 93.2 M                                                                                               
Total estimated model params size (MB): 372.783                                                                    
Modules in train mode: 249                                                                                         
Modules in eval mode: 0                                                                                            
Total FLOPs: 0

Output()

...

`Trainer.fit` stopped: `max_epochs=5` reached.


Restoring states from the checkpoint path at logs/timit_data2vec_finetune_val_12_layers/version_1/checkpoints/epoch=4-val_per=0.138.ckpt
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
Loaded model weights from the checkpoint at logs/timit_data2vec_finetune_val_12_layers/version_1/checkpoints/epoch=4-val_per=0.138.ckpt


Output()

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃      Validate metric      ┃       DataLoader 0        ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│         val_loss          │    0.39428457617759705    │
│          val_per          │    0.13486622273921967    │
└───────────────────────────┴───────────────────────────┘

[{'val_loss': 0.39428457617759705, 'val_per': 0.13486622273921967}]